# Alpha scheduled codegen: a Jupyter walkthrough

Mirrors `docs/scheduled-codegen-design.md` §5.2's own worked example end to end: parse a
prefix-sum program, normalize it (hoisting its `reduce` into its own pair of statements),
inspect it, attach an explicit target mapping (schedule), and generate C.

This notebook is also a regression fixture, checked with `nbval`
(`pytest --nbval alphalang/notebooks/prefix_sum.ipynb` from the repo root) — every cell's
saved output below is real, not illustrative; a code change that alters any of `alpha`'s
`__repr__` output or generated C will show up as a diff here.


In [1]:
import alphalang

## Reading the program

`%%alpha <var>` parses the cell body as Alpha source and binds an `alphalang.System` to `<var>`.

In [2]:
%%alpha sys
affine PrefixSum [N]->{:N>0}
    inputs  X: {[i]: 0<=i<N}
    outputs Y: {[i]: 0<=i<N}
    let 
        Y[i] = reduce(+, [j], {:j<=i}: X[j]);
.

`alphalang.print` is an indented debug tree dump of the whole AST — every node's own kind
plus its `expression_domain`/`context_domain`, ported from alpha-language's `PrintAST`.
Useful for seeing exactly what the parser/analyzer produced; `alphalang.show`/`alphalang.ashow`
(next) read back as actual Alpha-like source instead.


In [3]:
print(alphalang.print(sys))

System "PrefixSum"
    parameter_domain: [N] -> {  : N > 0 }
    inputs
        X : [N] -> { [i] : 0 <= i < N }
    outputs
        Y : [N] -> { [i] : 0 <= i < N }
    SystemBody[0] domain=[N] -> {  : N > 0 }
        StandardEquation Y index_names=["i"]
            Reduce operator=+ projection=[N] -> { [i, j] -> [(i)] } body_context=["i", "j"] exp=[N] -> { [i0] : N > 0 and i0 >= 0 } ctx=[N] -> { [i] : N > 0 and 0 <= i < N }
                Restrict domain=[N] -> { [i, j] : j <= i } exp=[N] -> { [i, j] : 0 <= j <= i and j < N } ctx=[N] -> { [i, j] : i < N and 0 <= j <= i }
                    Dependence function=[N] -> { [i, j] -> [(j)] } exp=[N] -> { [i, j] : 0 <= j < N } ctx=[N] -> { [i, j] : i < N and 0 <= j <= i and j < N }
                        Variable "X" exp=[N] -> { [i] : 0 <= i < N } ctx=[N] -> { [i0] : 0 <= i0 < N }



`alphalang.show` reconstructs Alpha-like source syntax from the model ("Show notation",
ported from `Show.xtend`) — a `Dependence` prints in point-free composition form
(`f@X`). `alphalang.ashow` (`AShow.xtend`) is the same idea in array-index notation
(`X[f]`), and also shows each equation's own ambient index names explicitly
(`Y[i] = ...`).


In [4]:
print(alphalang.show(sys))

affine PrefixSum [N] -> {  : N > 0 }
    inputs
        X : { [i] : 0 <= i < N }
    outputs
        Y : { [i] : 0 <= i < N }
    let
        Y = reduce(+, (i,j->i), { [i, j] : j <= i } : (i,j->j)@X);
.



In [5]:
print(alphalang.ashow(sys))

affine PrefixSum [N] -> {  : N > 0 }
    inputs
        X : { [i] : 0 <= i < N }
    outputs
        Y : { [i] : 0 <= i < N }
    let
        Y[i] = reduce(+, (i,j->i), { [i, j] : j <= i } : X[j]);
.



## Normalizing

`alphalang.normalize` splits `Y`'s equation into its own `Y__init`/`Y__reduce` statement pair
(§4.2) — `Y[i] = reduce(...)` is already its equation's own topmost node, so nothing needs
hoisting into a separate local first (see `docs/scheduled-codegen-design.md` §3) — and
returns a new `NormalizedSystem`; `sys` itself is untouched.


In [6]:
norm = alphalang.normalize(sys)
print(alphalang.show(norm))

affine PrefixSum [N] -> {  : N > 0 }
    inputs
        X : { [i] : 0 <= i < N }
    outputs
        Y : { [i] : 0 <= i < N }
    let
        Y = reduce(+, (i,j->i), { [i, j] : j <= i } : (i,j->j)@X);
.



## Scheduling

`%%schedule <var> <source-system-var>` parses its cell body as a target mapping (§6), validates
it, checks legality against `norm`'s real dependences (§7), and binds a new `ScheduledSystem`.

In [7]:
%%schedule sched norm
{ Y__init[i] -> [i, 0, 0]; Y__reduce[i,j] -> [i, 1, j]; }


In [8]:
print(sched)

[N] -> { Y__init[i] -> [i, 0, 0] : 0 <= i < N; Y__reduce[i, j] -> [i, 1, j] : i < N and 0 <= j <= i and j < N }


## Generating C

`alphalang.generate` runs `ScheduledC` and returns the generated C source as a `str`.

In [9]:
code = alphalang.generate(sched)
print(code)

// This code was auto-generated by alphac (alpha-rs).
// ScheduledC backend — a user-supplied target mapping controls loop order.

#include <float.h>
#include <limits.h>
#include <math.h>
#include <stdbool.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

// Function Macros
#define ceild(n,d) ((int)ceil(((double)(n))/((double)(d))))
#define floord(n,d) ((int)floor(((double)(n))/((double)(d))))
#define div(a,b) (ceild((a),(b)))
#define max(a,b) (((a)>(b))?(a):(b))
#define min(a,b) (((a)<(b))?(a):(b))
#define mallocCheck(v,s) if ((v) == NULL) { printf("Failed to allocate memory for variable: %s\n", (s)); exit(-1); }

// Global Variables
static long N;
static float* X;
static float* Y;

// Memory Macros
#define X(i0) X[i0]
#define Y(i0) Y[i0]

// Function Declarations
void PrefixSum(long _local_N, float* _local_X, float* _local_Y);

void PrefixSum(long _local_N, float* _local_X, float* _local_Y) {
	N = _local_N;
	X = _local_X;
	Y = _local_Y;

	// Allocate memory for local sto

## The legality check, exercised

An *omitted* target mapping means every statement gets its own identity schedule (§6) — for a
program with a real `reduce` dependency like this one, that's illegal: nothing guarantees
`Y__init`/`Y__reduce` run in the right order relative to each other. `%%schedule`
(and `alphalang.generate` on a bare `NormalizedSystem`) reject it with `alphalang.ScheduleError` rather
than silently generating wrong code.


In [10]:
try:
    alphalang.generate(norm)
except alphalang.ScheduleError as e:
    print(f"ScheduleError: {e}")

ScheduleError: 'Y__reduce' reads 'Y__init' but the schedule doesn't guarantee the producer instance runs strictly before the consumer instance that reads it (§7.2)
